In [0]:
# -------------------------------------------------------------------------
# DLT PIPELINE: BRONZE LAYER (01_bronze_ingestion)
# Purpose: Ingest raw CSV data using Auto Loader with Schema Evolution & Audit
# Rubric: Domain 3 (DLT, Streaming, Schema Evolution) & Domain 4 (Bronze)
# -------------------------------------------------------------------------

import dlt
from pyspark.sql.functions import current_timestamp, col, lit

# 1. Configuration (Hardcoded for clarity or use spark.conf.get for params)
# The path where your 109M rows are sitting
source_path = "/Volumes/ecommerce_analytics_dev/bronze_layer/raw_landing_volume/raw_landing"

# 2. Define the Bronze Table
@dlt.table(
    name="events_raw",
    comment="Raw ingestion table for e-commerce events with audit trails.",
    table_properties={
        "quality": "bronze",
        "delta.enableChangeDataFeed": "true" # Enables easier tracking for Silver
    }
)
def events_raw():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        # RUBRIC: Schema Evolution & Inference
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
        # RUBRIC: Error Handling / Dead Letter Queue (Quarantines bad data)
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .load(source_path)
        # RUBRIC: Audit Columns
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("created_at", current_timestamp())
    )